# Hieroglyph Detector Pipeline
This notebook uses the modular `detector` package. Core logic lives in:
- `config.py` — paths, transforms, constants
- `models.py` — ResNet50 classifier builder
- `segmenter.py` — SAM-based symbol segmentation (`HieroglyphSegmenter`)
- `classifier.py` — classification head (`HieroglyphClassifier`)
- `pipeline.py` — end-to-end image processing (`process_hieroglyphic_image`)
- `visualization.py` — plotting and result tables (`display_results`)

In [ ]:
%matplotlib inline
import cv2
from detector.config import DEVICE, SAM_CHECKPOINT, TARGET_IMAGE, idx_to_class, inference_transforms
from detector.models import load_classifier
from detector.segmenter import HieroglyphSegmenter
from detector.classifier import HieroglyphClassifier
from detector.pipeline import process_hieroglyphic_image
from detector.visualization import display_results

In [ ]:
print(f"Using device: {str(DEVICE).upper()}")
model = load_classifier()
print("System ready.\n")

In [ ]:
segmenter = HieroglyphSegmenter(
    sam_checkpoint_path=SAM_CHECKPOINT,
    config={
        'min_area': 250, 'max_area': 15000,
        'min_width': 10, 'min_height': 10,
        'max_width': 180, 'max_height': 180,
        'max_aspect_ratio': 6.0, 'min_aspect_ratio': 0.10,
        'min_solidity': 0.10, 'min_extent': 0.10,
        'crop_padding': 8, 'line_threshold': 55,
    },
)

classifier = HieroglyphClassifier(
    model=model,
    idx_to_class=idx_to_class,
    transform=inference_transforms,
    min_confidence=0.25,
)

In [ ]:
import os
if os.path.exists(TARGET_IMAGE):
    pipeline_result = process_hieroglyphic_image(
        image_path=TARGET_IMAGE,
        segmenter=segmenter,
        classifier=classifier,
        show_debug=True,
        max_entropy_ratio=0.75,
    )
    original_img = cv2.imread(TARGET_IMAGE)
    display_results(pipeline_result, original_img)

    stats = pipeline_result['stats']
    print("\nPipeline Performance Metrics:")
    seg_eff = stats['valid_contours'] / max(stats['total_contours'], 1) * 100
    cls_eff = stats['accepted'] / max(stats['final_crops'], 1) * 100
    tot_eff = stats['accepted'] / max(stats['total_contours'], 1) * 100
    print(f"  SAM Prompts Triggered: {stats['total_contours']}")
    print(f"  Valid Bounding Boxes: {stats['valid_contours']} ({seg_eff:.1f}%)")
    print(f"  Classification Head: {stats['accepted']}/{stats['final_crops']} ({cls_eff:.1f}%)")
    print(f"  Overall Pipeline Yield: {stats['accepted']}/{stats['total_contours']} ({tot_eff:.1f}%)")
else:
    print(f"Target image not found: {TARGET_IMAGE}")